In [6]:
# =============================================================================
# UNIFIED ROLLING-WINDOW MODEL COMPARISON PIPELINE
# =============================================================================

# ─────────────────────────────────────────────
# 1. IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# 2. LOAD & PREPARE DATA
# ─────────────────────────────────────────────
# FIX: Renamed global dataframe from `df` to `stock_df` to prevent
#      namespace collision with loop variables in the summary section.
stock_df = pd.read_csv("final_features_GOOGL_2022_2025.csv", parse_dates=["date"])
stock_df = stock_df.sort_values("date").reset_index(drop=True)

FEATURES = [
    "simple_return",
    "return_lag_1", "return_lag_2",
    "return_lag_5", "return_lag_10",
    "volume_lag_1", "volume_lag_5",
    "roc_10", "roc_20", "ewm_vol_10",
    "relative_return_1", "relative_strength_5", "relative_strength_20"
]
TARGET = "target_next_return"

X = stock_df[FEATURES]
y = stock_df[TARGET]
stock_df["month"] = stock_df["date"].dt.to_period("M")

test_predictions = []
# ─────────────────────────────────────────────
# 3. ROLLING WINDOW SPLITS (MONTH-BASED)
# ─────────────────────────────────────────────
TRAIN_WINDOW = 12  # months

def rolling_window_splits(months, train_window):
    # FIX: Added assertion to catch insufficient data early with a clear message
    #      instead of silently returning an empty fold list.
    assert len(months) > train_window + 1, (
        f"Not enough months ({len(months)}) for train_window={train_window}. "
        f"Need at least {train_window + 2} months."
    )
    folds = []
    for i in range(len(months) - train_window - 1):
        folds.append((
            months[i:i + train_window],
            months[i + train_window],
            months[i + train_window + 1]
        ))
    return folds

all_months = sorted(stock_df["month"].unique())
folds = rolling_window_splits(all_months, TRAIN_WINDOW)


# ─────────────────────────────────────────────
# 4. METRICS
# ─────────────────────────────────────────────
def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(y_true) == np.sign(y_pred))

# FIX: Sharpe is NO LONGER computed per fold here. Instead, raw strategy
#      returns are collected across all folds and Sharpe is computed once
#      on the full concatenated OOS series in `compute_aggregate_sharpe`.
#      Averaging per-fold Sharpes produces a statistically incorrect result.
def compute_aggregate_sharpe(all_true, all_pred, annualization=252):
    """
    Compute Sharpe ratio on the full concatenated out-of-sample return series.
    This is the correct method for walk-forward backtests — NOT averaging
    per-fold Sharpes.
    """
    positions = np.sign(all_pred)
    strategy_returns = positions * all_true
    std = np.std(strategy_returns)
    if std == 0:
        return np.nan
    return np.mean(strategy_returns) / std * np.sqrt(annualization)


# ─────────────────────────────────────────────
# 5. GENERIC ROLLING EVALUATION ENGINE
# ─────────────────────────────────────────────
def evaluate_model(model_name, model_builder):
    fold_results = []

    # Accumulators for concatenated OOS series (used for aggregate Sharpe)
    all_test_true = []
    all_test_pred = []

    print(f"\n{'='*100}")
    print(f"MODEL: {model_name}")
    print(f"{'='*100}")
    print(f"{'Fold':<5} {'Train':<24} {'Val':<10} {'Test':<10} "
          f"{'Val RMSE':>9} {'Test RMSE':>10} {'R²':>8}")

    for fold, (train_m, val_m, test_m) in enumerate(folds, start=1):
        train_idx = stock_df.index[stock_df["month"].isin(train_m)]
        val_idx   = stock_df.index[stock_df["month"] == val_m]
        test_idx  = stock_df.index[stock_df["month"] == test_m]

        # FIX: Guard against empty splits (e.g. months with no trading days).
        #      Without this, XGBoost early stopping throws a cryptic error and
        #      sklearn metrics silently produce NaN or raise exceptions.
        if len(train_idx) == 0 or len(val_idx) == 0 or len(test_idx) == 0:
            print(f"  Skipping fold {fold} — empty split "
                  f"(train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)})")
            continue

        X_train = X.loc[train_idx].values
        y_train = y.loc[train_idx].values
        X_val   = X.loc[val_idx].values
        y_val   = y.loc[val_idx].values
        X_test  = X.loc[test_idx].values
        y_test  = y.loc[test_idx].values

        # FIX: Removed StandardScaler — it is unnecessary for tree-based models
        #      (LightGBM, XGBoost, RandomForest are all invariant to feature
        #      scaling). Keeping it was wasteful and misleading to anyone who
        #      later adds a linear model and assumes the target is also scaled.

        model = model_builder()

        # -------------------------------
        # MODEL-SPECIFIC FIT LOGIC
        # -------------------------------
        if isinstance(model, lgb.LGBMRegressor):
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[
                    lgb.early_stopping(30, verbose=False),
                    lgb.log_evaluation(period=0)
                ]
            )

        elif isinstance(model, XGBRegressor):
            model.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                verbose=False
            )

        else:  # RandomForest
            model.fit(X_train, y_train)

        val_pred  = model.predict(X_val)
        test_pred = model.predict(X_test)

        # --------------------------------------------------
        # SAVE OOS PREDICTIONS (FOR ENSEMBLING)
        # --------------------------------------------------
        fold_pred = pd.DataFrame({
            "date": stock_df.loc[test_idx, "date"].values,
            "fold_id": fold,
            "y_true": y_test,
            "y_pred": test_pred,
            "model": model_name
        })
        
        test_predictions.append(fold_pred)

        # FIX: All metric calls now consistently use numpy arrays (.values was
        #      already applied above when slicing into X_train/y_train etc.)
        #      so there is no mixed Series/ndarray inconsistency.
        fold_results.append({
            "RMSE":                np.sqrt(mean_squared_error(y_test, test_pred)),
            "MAE":                 mean_absolute_error(y_test, test_pred),
            "R2":                  r2_score(y_test, test_pred),
            "Directional_Accuracy": directional_accuracy(y_test, test_pred),
        })

        # Accumulate OOS series for aggregate Sharpe
        all_test_true.extend(y_test)
        all_test_pred.extend(test_pred)

        print(f"{fold:<5} {str(train_m[0])+'→'+str(train_m[-1]):<24} "
              f"{str(val_m):<10} {str(test_m):<10} "
              f"{np.sqrt(mean_squared_error(y_val, val_pred)):>9.5f} "
              f"{np.sqrt(mean_squared_error(y_test, test_pred)):>10.5f} "
              f"{r2_score(y_test, test_pred):>8.4f}")

    results_df = pd.DataFrame(fold_results)

    # FIX: Compute Sharpe ONCE on the full concatenated OOS return series.
    #      This is the standard approach in quant finance for walk-forward
    #      backtests. The old approach (averaging monthly Sharpes) overstates
    #      the annualization effect and is not statistically valid.
    aggregate_sharpe = compute_aggregate_sharpe(
        np.array(all_test_true),
        np.array(all_test_pred)
    )
    results_df["Sharpe"] = np.nan  # placeholder per-fold (not meaningful)

    return results_df, aggregate_sharpe


# ─────────────────────────────────────────────
# 6. MODEL DEFINITIONS
# ─────────────────────────────────────────────
models = {
    "LightGBM": lambda: lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        num_leaves=15,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="regression",
        metric="rmse",
        n_jobs=-1,
        random_state=42,
        verbose=-1,
    ),

    "XGBoost": lambda: XGBRegressor(
        n_estimators= 500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,   # L1
        reg_lambda=1.0,   # L2
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=30,
        random_state=42,
        n_jobs=-1,
    ),

    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=300,
        max_depth=6,
        min_samples_leaf=10,
        min_samples_split= 20,
        max_features=0.5,
        max_samples=0.8,
        random_state=42,
        n_jobs=-1
    )
}


# ─────────────────────────────────────────────
# 7. RUN ALL MODELS
# ─────────────────────────────────────────────
all_results      = {}   # fold-level DataFrames
all_sharpes      = {}   # aggregate OOS Sharpe per model

for name, builder in models.items():
    res_df, agg_sharpe     = evaluate_model(name, builder)
    all_results[name]      = res_df
    all_sharpes[name]      = agg_sharpe

# ─────────────────────────────────────────────
# SAVE OOS PREDICTIONS FOR ENSEMBLING
# ─────────────────────────────────────────────
ml_test_df = (
    pd.concat(test_predictions, ignore_index=True)
      .sort_values(["date", "model"])
      .reset_index(drop=True)
)

# Explicit typing (prevents silent merge bugs later)
ml_test_df["fold_id"] = ml_test_df["fold_id"].astype(int)
ml_test_df["model"]   = ml_test_df["model"].astype(str)

ml_test_df.to_csv("ml_test_predictions.csv", index=False)

print(f"\nSaved ML test predictions: {ml_test_df.shape[0]:,} rows")
print("Columns:", list(ml_test_df.columns))


# ─────────────────────────────────────────────
# 8. COMPARISON SUMMARY TABLE (CLEAN & EXPLICIT)
# ─────────────────────────────────────────────
rows = []

for model, res_df in all_results.items():
    rows.append({
        "model":                       model,
        "RMSE_mean":                   res_df["RMSE"].mean(),
        "RMSE_std":                    res_df["RMSE"].std(),
        "MAE_mean":                    res_df["MAE"].mean(),
        "MAE_std":                     res_df["MAE"].std(),
        "R2_mean":                     res_df["R2"].mean(),
        "R2_std":                      res_df["R2"].std(),
        "Directional_Accuracy_mean":   res_df["Directional_Accuracy"].mean(),
        "Directional_Accuracy_std":    res_df["Directional_Accuracy"].std(),
        "Sharpe_aggregate":            all_sharpes[model],
    })

summary_df = pd.DataFrame(rows).set_index("model")

# Pretty multi-index columns (paper-ready)
summary_df.columns = pd.MultiIndex.from_tuples([
    ("RMSE",                 "mean"),
    ("RMSE",                 "std"),
    ("MAE",                  "mean"),
    ("MAE",                  "std"),
    ("R2",                   "mean"),
    ("R2",                   "std"),
    ("Directional_Accuracy", "mean"),
    ("Directional_Accuracy", "std"),
    ("Sharpe",               "aggregate_OOS"),   # clearly labelled
])

print("\n" + "="*75)
print("SUMMARY — fold-averaged metrics")
print("="*75)
print(summary_df.round(5).to_string())


# ─────────────────────────────────────────────
# 9. RANKINGS
# ─────────────────────────────────────────────
print("\n── Rankings (best model per metric) ──")

# (metric_key, multiindex_col, lower_is_better)
metric_specs = [
    ("RMSE",                 ("RMSE",                 "mean"),          True),
    ("MAE",                  ("MAE",                  "mean"),          True),
    ("R2",                   ("R2",                   "mean"),          False),
    ("Directional_Accuracy", ("Directional_Accuracy", "mean"),          False),
    ("Sharpe",               ("Sharpe",               "aggregate_OOS"), False),
]

for label, col, lower_is_better in metric_specs:
    if lower_is_better:
        best_model = summary_df[col].idxmin()
    else:
        best_model = summary_df[col].idxmax()

    best_value = summary_df.loc[best_model, col]
    print(f"  {label:<22}: {best_model} ({best_value:.5f})")


MODEL: LightGBM
Fold  Train                    Val        Test        Val RMSE  Test RMSE       R²
1     2022-01→2022-12          2023-01    2023-02      0.02204    0.03090  -0.0045
2     2022-02→2023-01          2023-02    2023-03      0.03110    0.02041  -0.1395
3     2022-03→2023-02          2023-03    2023-04      0.02051    0.01755  -0.0373
4     2022-04→2023-03          2023-04    2023-05      0.01727    0.01825  -0.1740
5     2022-05→2023-04          2023-05    2023-06      0.01795    0.01491  -0.0441
6     2022-06→2023-05          2023-06    2023-07      0.01430    0.02062  -0.0320
7     2022-07→2023-06          2023-07    2023-08      0.02071    0.01365   0.0075
8     2022-08→2023-07          2023-08    2023-09      0.01287    0.01375   0.0018
9     2022-09→2023-08          2023-09    2023-10      0.01388    0.02439  -0.0178
10    2022-10→2023-09          2023-10    2023-11      0.02436    0.01072   0.0058
11    2022-11→2023-10          2023-11    2023-12      0.00966    0.01